In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# List folder contents
import os

folder_path = "/content/drive/MyDrive/Colab Notebooks/ColabData"
print(os.listdir(folder_path))

In [ ]:
# Extract the zip file
import zipfile

zip_path = "/content/drive/MyDrive/Colab Notebooks/ColabData/motion-and-heart-rate-from-a-wrist-worn-wearable-and-labeled-sleep-from-polysomnography-1.0.0.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/sleep_accel_data")

print("Done extracting")
print(os.listdir("/content/sleep_accel_data"))

In [ ]:
# Check if directory exists
DATA_DIR = "/content/sleep_accel_data/motion-and-heart-rate-from-a-wrist-worn-wearable-and-labeled-sleep-from-polysomnography-1.0.0"
print(os.path.exists(DATA_DIR))

In [ ]:
# Count files in each subdirectory
import os

for sub in os.listdir(DATA_DIR):
    path = os.path.join(DATA_DIR, sub)
    if os.path.isdir(path):
        n_files = len(os.listdir(path))
        print(f"{sub}/  -> {n_files} files")

In [ ]:
# Sample files from each folder
import glob

motion_files = glob.glob(f"{DATA_DIR}/motion/*")
hr_files = glob.glob(f"{DATA_DIR}/heart_rate/*")
label_files = glob.glob(f"{DATA_DIR}/labels/*")

print("Sample motion file:", motion_files[0])
!head -3 "{motion_files[0]}"
print()
print("Sample heart rate file:", hr_files[0])
!head -3 "{hr_files[0]}"
print()
print("Sample label file:", label_files[0])
!head -3 "{label_files[0]}"

In [ ]:
# Extract subject IDs from label filenames
subject_ids = sorted([os.path.basename(f).split("_")[0] for f in label_files])
print(f"Found {len(subject_ids)} subjects")
print(subject_ids[:5])

In [ ]:
# Loading data and computing epoch features
import pandas as pd

def load_labels(subject_id):
    path = f"{DATA_DIR}/labels/{subject_id}_labeled_sleep.txt"
    return pd.read_csv(path, sep=" ", header=None, names=["time", "stage"])

def load_motion(subject_id):
    path = f"{DATA_DIR}/motion/{subject_id}_acceleration.txt"
    return pd.read_csv(path, sep=" ", header=None, names=["time", "x", "y", "z"])

def load_heart_rate(subject_id):
    path = f"{DATA_DIR}/heart_rate/{subject_id}_heartrate.txt"
    return pd.read_csv(path, header=None, names=["time", "hr"])  # comma by default

def load_steps(subject_id):
    path = f"{DATA_DIR}/steps/{subject_id}_steps.txt"
    return pd.read_csv(path, header=None, names=["time", "steps"])

# Sanity check
sid = subject_ids[0]
print(load_labels(sid).head())
print(load_motion(sid).head())
print(load_heart_rate(sid).head())
print(load_steps(sid).head())

In [ ]:
# Loading data and computing epoch features
import numpy as np

def epoch_features(subject_id):
    labels_df = load_labels(subject_id)
    motion_df = load_motion(subject_id)
    hr_df = load_heart_rate(subject_id)
    steps_df = load_steps(subject_id)

    rows = []
    for _, row in labels_df.iterrows():
        t0, t1 = row["time"], row["time"] + 30

        m_win = motion_df[(motion_df.time >= t0) & (motion_df.time < t1)]
        hr_win = hr_df[(hr_df.time >= t0) & (hr_df.time < t1)]
        st_win = steps_df[(steps_df.time >= t0) & (steps_df.time < t1)]

        if len(m_win) > 0:
            mag = np.sqrt(m_win.x**2 + m_win.y**2 + m_win.z**2)
            acc_mean, acc_std, acc_max = mag.mean(), mag.std(), mag.max()
        else:
            acc_mean = acc_std = acc_max = np.nan

        hr_mean = hr_win.hr.mean() if len(hr_win) > 0 else np.nan
        steps_sum = st_win.steps.sum() if len(st_win) > 0 else 0

        rows.append({
            "subject": subject_id, "time": row["time"], "stage": row["stage"],
            "acc_mean": acc_mean, "acc_std": acc_std, "acc_max": acc_max,
            "hr_mean": hr_mean, "steps_sum": steps_sum
        })
    return pd.DataFrame(rows)


test_df = epoch_features(sid)
print(test_df.shape)
test_df.head()

In [ ]:
# build full_df
all_dfs = []
for s in subject_ids:
    try:
        df = epoch_features(s)
        all_dfs.append(df)
        print(f"{s}: {len(df)} epochs")
    except Exception as e:
        print(f"Skipping {s}: {e}")

full_df = pd.concat(all_dfs, ignore_index=True)
full_df.to_csv("/content/sleep_accel_epoched.csv", index=False)
print(full_df.shape)
full_df.head()

In [ ]:
#  NA counts / stage distribution
print(full_df.isna().sum())
print()
print(full_df.stage.value_counts().sort_index())

In [ ]:
# clean_df
clean_df = full_df.dropna(subset=["acc_mean", "hr_mean"]).copy()
print("Before:", full_df.shape, "After:", clean_df.shape)

In [ ]:
# PLACEHOLDER — will confirm/adjust based on your value_counts() output
stage_map = {
    -1: "Unscored",
    0: "Wake",
    1: "N1",
    2: "N2",
    3: "N3",
    4: "N3",
    5: "REM"
}
clean_df["stage_label"] = clean_df["stage"].map(stage_map).fillna("Other")
print(clean_df.stage_label.value_counts())

In [ ]:
# filter out "Unscored" epochs
eda_df = clean_df[clean_df.stage_label != "Unscored"].copy()
print(eda_df.shape)
print(eda_df.stage_label.value_counts())

In [ ]:
# per-subject epoch counts
print("Subjects:", eda_df.subject.nunique())
print("Total epochs:", len(eda_df))
print()
print(eda_df.groupby("subject").size().describe())

In [ ]:
# sleep stage distribution
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

stage_order = ["Wake", "N1", "N2", "N3", "REM"]

plt.figure(figsize=(8,5))
sns.countplot(data=eda_df, x="stage_label", order=stage_order)
plt.title("Distribution of Sleep Stages Across All Subjects")
plt.xlabel("Sleep Stage")
plt.ylabel("Number of 30s Epochs")
plt.show()

In [ ]:
# HR and motion by sleep stage
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=eda_df, x="stage_label", y="hr_mean", order=stage_order, ax=axes[0])
axes[0].set_title("Heart Rate by Sleep Stage")
axes[0].set_xlabel("Sleep Stage")
axes[0].set_ylabel("Heart Rate (bpm)")

sns.boxplot(data=eda_df, x="stage_label", y="acc_mean", order=stage_order, ax=axes[1])
axes[1].set_title("Motion (Accel. Magnitude) by Sleep Stage")
axes[1].set_xlabel("Sleep Stage")
axes[1].set_ylabel("Mean Acceleration Magnitude (g)")

plt.tight_layout()
plt.show()

In [ ]:
# hypnogram for one subject
sid_example = eda_df.subject.unique()[0]
sub_df = eda_df[eda_df.subject == sid_example].sort_values("time")

plt.figure(figsize=(14, 3))
plt.step(sub_df.time / 3600, sub_df.stage, where="post")
plt.xlabel("Time (hours, relative)")
plt.ylabel("Stage code")
plt.title(f"Hypnogram — Subject {sid_example}")
plt.gca().invert_yaxis()  # convention: deeper sleep stages plotted lower
plt.show()

In [ ]:
# feature correlation matrix
corr = eda_df[["acc_mean", "acc_std", "acc_max", "hr_mean", "steps_sum"]].corr()

plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Feature Correlation Matrix")
plt.show()

In [ ]:
# sleep stage composition per subject
stage_time = eda_df.groupby(["subject", "stage_label"]).size().unstack(fill_value=0)
stage_time = stage_time[[c for c in stage_order if c in stage_time.columns]]

stage_time_pct = stage_time.div(stage_time.sum(axis=1), axis=0) * 100

stage_time_pct.plot(kind="bar", stacked=True, figsize=(16,6), colormap="viridis")
plt.title("Sleep Stage Composition per Subject (%)")
plt.xlabel("Subject")
plt.ylabel("Percentage of Night")
plt.legend(title="Stage", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Sanity check: confirm eda_df is the corrected, current version
print(eda_df["stage_label"].value_counts())
print()
print(eda_df.groupby("subject")["stage_label"].value_counts(normalize=True)
      .unstack(fill_value=0)["REM"].describe())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- Per-subject class distribution (as percentages, easier to compare across subjects) ---
stage_order = ["Wake", "N1", "N2", "N3", "REM"]

per_subject_stage = eda_df.groupby(["subject", "stage_label"]).size().unstack(fill_value=0)
per_subject_stage = per_subject_stage[[c for c in stage_order if c in per_subject_stage.columns]]
per_subject_stage_pct = per_subject_stage.div(per_subject_stage.sum(axis=1), axis=0) * 100

print("Per-subject stage distribution (%):")
print(per_subject_stage_pct.round(1))

# Heatmap view — quickly spot subjects with unusual stage balance
plt.figure(figsize=(10, 10))
sns.heatmap(per_subject_stage_pct, annot=True, fmt=".0f", cmap="YlGnBu", cbar_kws={'label': '% of night'})
plt.title("Sleep Stage Distribution per Subject (%)")
plt.xlabel("Sleep Stage")
plt.ylabel("Subject")
plt.tight_layout()
plt.show()

# --- Recording length distribution ---
# Each epoch = 30 seconds, so epoch count directly converts to hours
recording_length = eda_df.groupby("subject").size() * 30 / 3600  # hours

print("\nRecording length (hours) summary:")
print(recording_length.describe())

plt.figure(figsize=(8,5))
sns.histplot(recording_length, bins=15, kde=True)
plt.xlabel("Recording Length (hours)")
plt.ylabel("Number of Subjects")
plt.title("Distribution of Recording Length Across Subjects")
plt.show()

# Flag any unusually short recordings — worth a second look
short_subjects = recording_length[recording_length < recording_length.quantile(0.1)]
print("\nShortest recordings (bottom 10%):")
print(short_subjects)

In [ ]:
import numpy as np
import matplotlib.patches as mpatches

def plot_raw_night(subject_id):
    labels_df = load_labels(subject_id)
    motion_df = load_motion(subject_id)
    hr_df = load_heart_rate(subject_id)

    # Compute motion magnitude from raw x,y,z
    motion_df = motion_df.copy()
    motion_df["mag"] = np.sqrt(motion_df.x**2 + motion_df.y**2 + motion_df.z**2)

    # Color map for stages
    stage_colors = {
        -1: "#dddddd",  # Unscored
        0: "#f4a582",   # Wake
        1: "#92c5de",   # N1
        2: "#4393c3",   # N2
        3: "#2166ac",   # N3
        4: "#b2182b"    # REM
    }
    stage_names = {-1: "Unscored", 0: "Wake", 1: "N1", 2: "N2", 3: "N3", 4: "REM"}

    fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)

    # Shade background by sleep stage using the label epochs
    for _, row in labels_df.iterrows():
        t0, t1 = row.time, row.time + 30
        color = stage_colors.get(row.stage, "#ffffff")
        for ax in axes:
            ax.axvspan(t0/3600, t1/3600, color=color, alpha=0.5, linewidth=0)

    # Heart rate trace
    axes[0].plot(hr_df.time/3600, hr_df.hr, color="black", linewidth=0.8)
    axes[0].set_ylabel("Heart Rate (bpm)")
    axes[0].set_title(f"Subject {subject_id} — Raw Heart Rate & Motion, Colored by Sleep Stage")
    axes[0].set_xlim(labels_df.time.min()/3600 - 0.5, labels_df.time.max()/3600 + 0.5)

    # Motion magnitude trace
    axes[1].plot(motion_df.time/3600, motion_df.mag, color="black", linewidth=0.3)
    axes[1].set_ylabel("Accel. Magnitude (g)")
    axes[1].set_xlabel("Time (hours, relative)")

    # Legend
    handles = [mpatches.Patch(color=stage_colors[k], label=stage_names[k], alpha=0.5)
               for k in stage_colors if k in labels_df.stage.unique()]
    axes[0].legend(handles=handles, loc="upper right", ncol=len(handles), fontsize=8)

    plt.tight_layout()
    plt.show()

# Try it on a subject — pick one with a reasonably full night
plot_raw_night(subject_ids[0])

In [ ]:
# --- Inspect HR for implausible values ---
print("HR distribution:")
print(eda_df.hr_mean.describe())

# Physiologically plausible resting/sleep HR range — anything way outside this
# is likely a PPG sensor artifact (motion during sleep is a known cause)
plt.figure(figsize=(8,5))
sns.histplot(eda_df.hr_mean, bins=50, kde=True)
plt.axvline(30, color="red", linestyle="--", label="Lower bound (30 bpm)")
plt.axvline(180, color="red", linestyle="--", label="Upper bound (180 bpm)")
plt.title("Heart Rate Distribution — Checking for Implausible Values")
plt.xlabel("Mean HR per epoch (bpm)")
plt.legend()
plt.show()

n_low = (eda_df.hr_mean < 30).sum()
n_high = (eda_df.hr_mean > 180).sum()
print(f"Epochs with HR < 30 bpm: {n_low}")
print(f"Epochs with HR > 180 bpm: {n_high}")

# --- Inspect accelerometer magnitude for spikes ---
print("\nAcceleration magnitude distribution:")
print(eda_df.acc_max.describe())

plt.figure(figsize=(8,5))
sns.histplot(eda_df.acc_max, bins=50, kde=True)
plt.title("Max Acceleration Magnitude per Epoch — Checking for Spikes")
plt.xlabel("Max accel magnitude (g)")
plt.show()

# Anything wildly above 1g (gravity at rest) for sustained periods could be
# a sensor glitch rather than real movement — flag extreme outliers using IQR
q1, q3 = eda_df.acc_max.quantile([0.25, 0.75])
iqr = q3 - q1
upper_fence = q3 + 3 * iqr  # using 3xIQR (conservative) instead of the usual 1.5x
n_outliers = (eda_df.acc_max > upper_fence).sum()
print(f"\nAccel outlier threshold (3xIQR): {upper_fence:.2f} g")
print(f"Epochs flagged as accel outliers: {n_outliers}")

# --- Apply clipping (only if you're confident these are artifacts, not real data) ---
eda_df["hr_mean_clipped"] = eda_df.hr_mean.clip(30, 180)
eda_df["acc_max_clipped"] = eda_df.acc_max.clip(upper=upper_fence)

print("\nAfter clipping:")
print(eda_df[["hr_mean_clipped", "acc_max_clipped"]].describe())

In [ ]:
# --- Steps distribution overall ---
print("Steps per epoch summary:")
print(eda_df.steps_sum.describe())

plt.figure(figsize=(8,5))
sns.histplot(eda_df[eda_df.steps_sum > 0].steps_sum, bins=40)
plt.title("Distribution of Step Counts (non-zero epochs only)")
plt.xlabel("Steps per 30s epoch")
plt.show()

# --- What % of epochs have any steps at all, and how does that vary by stage? ---
eda_df["has_steps"] = eda_df.steps_sum > 0
pct_with_steps = eda_df.groupby("stage_label")["has_steps"].mean() * 100

plt.figure(figsize=(8,5))
pct_with_steps.reindex(stage_order).plot(kind="bar", color="teal")
plt.title("% of Epochs With Any Steps Recorded, by Sleep Stage")
plt.ylabel("% of epochs")
plt.xlabel("Sleep Stage")
plt.show()

print(pct_with_steps)

In [ ]:
from scipy.stats import kruskal

stage_order = ["Wake", "N1", "N2", "N3", "REM"]

# Kruskal-Wallis is the non-parametric alternative to ANOVA — appropriate here
# since HR distributions per stage are unlikely to be normally distributed
groups = [eda_df[eda_df.stage_label == stage].hr_mean.dropna() for stage in stage_order if stage in eda_df.stage_label.unique()]

h_stat, p_value = kruskal(*groups)
print(f"Kruskal-Wallis H-statistic: {h_stat:.2f}")
print(f"p-value: {p_value:.2e}")

if p_value < 0.05:
    print("\n→ Significant difference in HR distributions across sleep stages.")
else:
    print("\n→ No significant difference detected.")

# Optional: pairwise post-hoc test (Dunn's test) to see WHICH stages differ from each other
!pip install -q scikit-posthocs
import scikit_posthocs as sp

dunn_result = sp.posthoc_dunn(eda_df, val_col="hr_mean", group_col="stage_label", p_adjust="bonferroni")
print("\nPairwise comparison (Dunn's test, Bonferroni-corrected p-values):")
print(dunn_result.round(4))

In [ ]:
# CSV — universally readable, but loses exact dtypes (e.g. int vs float) on reload
eda_df.to_csv("/content/sleep_accel_preprocessed.csv", index=False)

# Parquet — preserves dtypes, smaller file, faster to load
eda_df.to_parquet("/content/sleep_accel_preprocessed.parquet", index=False)

print("Both files saved")

In [ ]:
from google.colab import files

files.download("/content/sleep_accel_preprocessed.csv")
files.download("/content/sleep_accel_preprocessed.parquet")

**FEATURE ENGINEERING**

In [ ]:
def motion_features(m_win):
    if len(m_win) == 0:
        return dict.fromkeys(
            ["acc_mean","acc_std","acc_max","acc_min","acc_energy",
             "acc_iqr","acc_zcr","x_std","y_std","z_std"], np.nan)
    mag = np.sqrt(m_win.x**2 + m_win.y**2 + m_win.z**2)
    return {
        "acc_mean": mag.mean(), "acc_std": mag.std(),
        "acc_max": mag.max(), "acc_min": mag.min(),
        "acc_energy": (mag**2).mean(),                 # overall movement intensity
        "acc_iqr": mag.quantile(0.75) - mag.quantile(0.25),
        "acc_zcr": ((mag.diff().fillna(0) > 0).astype(int).diff().abs() > 0).mean(),  # zero-crossing-ish activity rate
        "x_std": m_win.x.std(), "y_std": m_win.y.std(), "z_std": m_win.z.std(),  # axis-wise variability
    }

In [ ]:
def hr_features(hr_win):
    if len(hr_win) == 0:
        return dict.fromkeys(["hr_mean","hr_std","hr_min","hr_max","hr_range"], np.nan)
    return {
        "hr_mean": hr_win.hr.mean(), "hr_std": hr_win.hr.std(),
        "hr_min": hr_win.hr.min(), "hr_max": hr_win.hr.max(),
        "hr_range": hr_win.hr.max() - hr_win.hr.min(),
    }

In [ ]:
def add_temporal_features(df):
    df = df.sort_values(["subject", "time"]).copy()
    # rolling context: mean HR/accel over the last 5 min (10 epochs) and next 5 min
    for col in ["hr_mean_clipped", "acc_mean"]:
        df[f"{col}_roll5_mean"] = df.groupby("subject")[col].transform(
            lambda s: s.rolling(5, min_periods=1).mean())
        df[f"{col}_roll5_std"] = df.groupby("subject")[col].transform(
            lambda s: s.rolling(5, min_periods=1).std())
    # delta from previous epoch — captures transitions (e.g. HR drop into deeper sleep)
    df["hr_delta"] = df.groupby("subject")["hr_mean_clipped"].diff()
    df["acc_delta"] = df.groupby("subject")["acc_mean"].diff()
    # position in the night — useful since REM increases later in the night
    df["epoch_idx"] = df.groupby("subject").cumcount()
    df["time_since_onset_hr"] = df["time"] / 3600
    # previous epoch's stage — strong predictor, but only usable if the model architecture allows it
    return df

In [ ]:
import pandas as pd
eda_df = pd.read_parquet("/content/sleep_accel_preprocessed.parquet")
print(eda_df.shape)
print(eda_df.columns.tolist())

In [ ]:
eda_df["hr_mean_clipped"] = eda_df["hr_mean"].clip(30, 180)

q1, q3 = eda_df["acc_max"].quantile([0.25, 0.75])
iqr = q3 - q1
upper_fence = q3 + 3 * iqr
eda_df["acc_max_clipped"] = eda_df["acc_max"].clip(upper=upper_fence)

print(eda_df[["hr_mean_clipped", "acc_max_clipped"]].describe())

In [ ]:
def add_temporal_features(df):
    df = df.sort_values(["subject", "time"]).copy()
    for col in ["hr_mean_clipped", "acc_mean"]:
        df[f"{col}_roll5_mean"] = df.groupby("subject")[col].transform(
            lambda s: s.rolling(5, min_periods=1).mean())
        df[f"{col}_roll5_std"] = df.groupby("subject")[col].transform(
            lambda s: s.rolling(5, min_periods=1).std())
    df["hr_delta"] = df.groupby("subject")["hr_mean_clipped"].diff()
    df["acc_delta"] = df.groupby("subject")["acc_mean"].diff()
    df["epoch_idx"] = df.groupby("subject").cumcount()
    df["time_since_onset_hr"] = df["time"] / 3600
    return df

eda_df = add_temporal_features(eda_df)
print(eda_df.columns.tolist())

In [ ]:
# Per-subject normalization (removes person-level baseline differences)
for col in ["hr_mean_clipped", "acc_mean"]:
    eda_df[f"{col}_subj_z"] = eda_df.groupby("subject")[col].transform(
        lambda s: (s - s.mean()) / (s.std() + 1e-6))

# Steps: binary flag + rolling burst window (raw steps_sum is ~96% zero)
eda_df["steps_roll5_sum"] = eda_df.groupby("subject")["steps_sum"].transform(
    lambda s: s.rolling(5, min_periods=1).sum())
# has_steps already exists per your last screenshot — skip re-adding it

# Night position: normalized 0-1 position through each subject's recording
eda_df["night_position"] = eda_df.groupby("subject")["time_since_onset_hr"].transform(
    lambda s: (s - s.min()) / (s.max() - s.min() + 1e-6))

print(eda_df.columns.tolist())
print(eda_df[["hr_mean_clipped_subj_z", "acc_mean_subj_z",
              "steps_roll5_sum", "night_position"]].describe())

In [ ]:
from sklearn.model_selection import train_test_split

all_subjects = eda_df["subject"].unique()
train_subjects, test_subjects = train_test_split(
    all_subjects, test_size=0.2, random_state=42
)

train_df = eda_df[eda_df["subject"].isin(train_subjects)].copy()
test_df = eda_df[eda_df["subject"].isin(test_subjects)].copy()

print(f"Train subjects: {len(train_subjects)}  ({len(train_df)} epochs)")
print(f"Test subjects: {len(test_subjects)}  ({len(test_df)} epochs)")

In [ ]:
print(train_df.columns.tolist())
print("hr_mean_clipped_subj_z" in train_df.columns)
print("night_position" in train_df.columns)

In [ ]:
feature_cols = [
    "acc_mean", "acc_std", "acc_max", "acc_max_clipped",
    "hr_mean_clipped",
    "hr_mean_clipped_roll5_mean", "hr_mean_clipped_roll5_std",
    "acc_mean_roll5_mean", "acc_mean_roll5_std",
    "hr_delta", "acc_delta",
    "hr_mean_clipped_subj_z", "acc_mean_subj_z",
    "has_steps", "steps_roll5_sum",
    "night_position",
]

In [ ]:
print(train_df[feature_cols].isna().sum())

In [ ]:
train_df[feature_cols] = train_df[feature_cols].fillna(0)
test_df[feature_cols] = test_df[feature_cols].fillna(0)

# confirm it's clean
print(train_df[feature_cols].isna().sum().sum())
print(test_df[feature_cols].isna().sum().sum())

In [ ]:
from sklearn.preprocessing import StandardScaler

feature_cols = [
    "acc_mean", "acc_std", "acc_max", "acc_max_clipped",
    "hr_mean_clipped",
    "hr_mean_clipped_roll5_mean", "hr_mean_clipped_roll5_std",
    "acc_mean_roll5_mean", "acc_mean_roll5_std",
    "hr_delta", "acc_delta",
    "hr_mean_clipped_subj_z", "acc_mean_subj_z",
    "has_steps", "steps_roll5_sum",
    "night_position",
]

missing = [c for c in feature_cols if c not in train_df.columns]
if missing:
    print("Dropping columns not found in train_df:", missing)
feature_cols = [c for c in feature_cols if c in train_df.columns]

# fillna before scaling (from the NaN check step)
train_df[feature_cols] = train_df[feature_cols].fillna(0)
test_df[feature_cols] = test_df[feature_cols].fillna(0)

scaler = StandardScaler().fit(train_df[feature_cols])
train_df[feature_cols] = scaler.transform(train_df[feature_cols])
test_df[feature_cols] = scaler.transform(test_df[feature_cols])

print("Features used:", feature_cols)
print(f"Total: {len(feature_cols)} features")
train_df[feature_cols].describe()

In [ ]:
stage_order = ["Wake", "N1", "N2", "N3", "REM"]

print("Train:")
print((train_df["stage_label"].value_counts(normalize=True)*100).reindex(stage_order).round(1))
print("\nTest:")
print((test_df["stage_label"].value_counts(normalize=True)*100).reindex(stage_order).round(1))

In [ ]:
# Feature distribution per class
n_feats = len(feature_cols)
fig, axes = plt.subplots((n_feats+2)//3, 3, figsize=(15, 4*((n_feats+2)//3)))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    sns.boxplot(data=train_df, x="stage_label", y=col, order=stage_order, ax=axes[i])
    axes[i].set_title(col, fontsize=10)
    axes[i].tick_params(axis='x', rotation=45)

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

In [ ]:
# Co-relation matrix

plt.figure(figsize=(12,10))
corr = train_df[feature_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Feature Correlation Matrix (16 features)")
plt.tight_layout()
plt.show()

high_corr = corr.abs().unstack().sort_values(ascending=False)
high_corr = high_corr[(high_corr < 1.0) & (high_corr > 0.85)]
print(high_corr.drop_duplicates())

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_probe = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                    random_state=42, n_jobs=-1)
rf_probe.fit(train_df[feature_cols], train_df["stage_label"])

importances = pd.Series(rf_probe.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)

In [ ]:
feature_cols = [
    "acc_mean", "acc_std",
    "hr_mean_clipped_roll5_mean", "hr_mean_clipped_roll5_std",
    "acc_mean_roll5_mean", "acc_mean_roll5_std",
    "hr_delta", "acc_delta",
    "hr_mean_clipped_subj_z", "acc_mean_subj_z",
    "steps_roll5_sum",
    "night_position",
]
print("Final feature set:", feature_cols)
print("Count:", len(feature_cols))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json

train_df.to_parquet("/content/drive/MyDrive/Colab Notebooks/ColabData/frozen/train_final.parquet")
test_df.to_parquet("/content/drive/MyDrive/Colab Notebooks/ColabData/frozen/test_final.parquet")
json.dump(feature_cols, open("/content/drive/MyDrive/Colab Notebooks/ColabData/frozen/feature_cols_final.json", "w"))

print("Saved.")